# GitHub repositories and users recommendations by embeddings

## Problem Statement

Currently, GitHub has two possibilities to explore users and repositories:
1. Direct search by search term leveraging names and tags.
2. Recommender system under 'Explore' tab which gives suggestions to a user based on his usage of service.

However, there is no possibility to perform a search of connected entities. E.g., find repositories or users highly related to each other.

## Goal of the Project

The goal of this project is to build GitHub repository search/recommender system, which would allow exploring connected repositories and people, by leveraging the underlying graph structure of the repositories database.

## Implemented ML solution

It was decided to build graph nodes embeddings (`repo2vec` and `user2vec`) for the entire GitHub database using [PyTorch-BigGraph (PBG)](https://github.com/facebookresearch/PyTorch-BigGraph). On top of the embeddings representation, we have built query tool with the ranking engine.

# Pipeline

In [1]:
from resources.utils import *

DUMP_DATE = '2021-03-06'
DUMP_DATE

'2021-03-06'

## Dataset: preprocessing

The goal of this stage is to process the GitHub dataset from http://ghtorrent.org/ project into directed graph. <br>
The source data consists from .csv files which represent tabular SQL data. Each file is one table in source database. <br>
Total size of the dataset is 100GB. The size of the tables used for graph building is ~7GB.

*Database schema:*
![image.png](http://ghtorrent.org/files/schema.png)

The following tables are used for the graph representation:
* Followers
* Watchers
* Project Members

##### Graph structure 
Following nodes are defined as points of interest for this project:
* Users
* Projects

Types of graph edges:
* Follows
* Watches
* Is member of

Possible edges:
* User => follow => User
* User => follow => Project
* User => watch => User
* User => watch => Project
* User => is member => Project

### Download data (http://ghtorrent.org/)

Run `db_download.sh` script (at terminal) to download dump

Extract relationships tables (CSV): `followers.csv`, `watchers.csv`, `project_members.csv`

and metadata: `users.csv`, `projects.csv`

to `data/mysql-%Y-%m-%d` folder (or change directories code dependencies at notebooks)

In [2]:
!tar -C data/ -xvzf data/mysql-{DUMP_DATE}.tar.gz mysql-{DUMP_DATE}-01/project_members.csv mysql-{DUMP_DATE}-01/followers.csv mysql-{DUMP_DATE}-01/watchers.csv mysql-{DUMP_DATE}-01/projects.csv mysql-{DUMP_DATE}/users.csv

mysql-2019-06-01/followers.csv
mysql-2019-06-01/watchers.csv
mysql-2019-06-01/projects.csv
mysql-2019-06-01/project_members.csv
mysql-2019-06-01/users.csv


In [7]:
!tar -ztvf data/mysql-{DUMP_DATE}.tar.gz

-rw-rw-rw- mysql/mysql       0 2021-03-07 02:04 dump/schema_info.csv
-rw-rw-rw- mysql/mysql       0 2021-03-07 02:04 dump/repo_milestones.csv
-rw-rw-r-- gousiosg/gousiosg 310 2021-03-07 02:18 dump/ORDER
-rw-rw-r-- gousiosg/gousiosg 703 2021-03-07 02:18 dump/indexes.sql
-rwxrwxr-x gousiosg/gousiosg 2882 2021-03-07 02:18 dump/ght-restore-mysql
-rw-rw-r-- gousiosg/gousiosg 6582 2021-03-07 02:18 dump/README.md
-rw-rw-r-- gousiosg/gousiosg 18979 2021-03-07 02:18 dump/schema.sql
-rw-rw-rw- mysql/mysql    35344340 2021-03-06 15:54 dump/organization_members.csv
-rw-rw-rw- mysql/mysql   109497299 2021-03-07 01:02 dump/project_topics.csv
-rw-rw-rw- mysql/mysql   600092193 2021-03-06 15:20 dump/dupl_projects.csv
-rw-rw-rw- mysql/mysql   607999615 2021-03-07 01:02 dump/project_members.csv
-rw-rw-rw- mysql/mysql   741475599 2021-03-06 15:49 dump/issue_labels.csv
-rw-rw-rw- mysql/mysql  1118734915 2021-03-06 15:21 dump/followers.csv
-rw-rw-rw- mysql/mysql  1795250426 2021-03-06 10:19 dump/commit_com

In [10]:
!tar -C data/ -xvzf data/mysql-{DUMP_DATE}.tar.gz dump/watchers.csv

dump/watchers.csv


In [9]:
!tar -C data/ -xvzf data/mysql-{DUMP_DATE}.tar.gz dump/followers.csv

dump/followers.csv


In [8]:
!tar -C data/ -xvzf data/mysql-{DUMP_DATE}.tar.gz dump/project_members.csv

dump/project_members.csv


### Process SQL structured data into edges

In [2]:
from pyspark import SparkContext, SparkConf, SQLContext

# constants
DATA_FOLDER = f"./data/mysql-{DUMP_DATE}/"
FOLLOWERS_PATH = DATA_FOLDER + "followers.csv"
WATCHERS_PATH = DATA_FOLDER + "watchers.csv"
PROJECT_MEMBERS_PATH = DATA_FOLDER + "project_members.csv"

# relations constants
IS_MEMBER_OF = "is_member_of"
FOLLOWS = "follows"
WATCHES = "watches"

In [3]:
def addIdentifiers(row, id_1, id_2):
    '''
    Adds identifier for each id to distinguish them
    '''
    row[0] = id_1 + row[0]
    row[1] = id_2 + row[1]
    return row

def swapColumns(row):
    temp = row[0]
    row[0] = row[1]
    row[1] = temp
    return row

def addRelation(row, relation):
    temp = row[1]
    row[1] = relation
    row.append(temp)
    return row

In [4]:
conf = SparkConf()
sc = SparkContext(conf=conf)
sc

<SparkContext master=local[*] appName=pyspark-shell>

Processing of members table

In [15]:
!head {PROJECT_MEMBERS_PATH}

1,1,"2009-12-08 10:17:27"
1,2,"2009-12-08 10:17:27"
1,4,"2009-12-08 10:17:27"
3,24,"2011-02-19 01:58:42"
3,5465,"2011-02-19 01:58:42"
3,11834,"2011-02-19 01:58:42"
3,11983,"2011-05-25 18:18:23"
3,17241,"2011-02-19 01:58:42"
3,26333,"2011-02-19 01:58:42"
3,45086,"2011-02-19 01:58:42"


In [8]:
members_rdd = sc.textFile(PROJECT_MEMBERS_PATH).map(lambda x: x.split(",")[:2])

members_rdd = members_rdd.map(
    lambda x: addIdentifiers(x, "repo_id_", "user_id_")
).map(swapColumns).map(lambda x: addRelation(x, IS_MEMBER_OF))
members_rdd.take(5)

[['user_id_1', 'is_member_of', 'repo_id_1'],
 ['user_id_2', 'is_member_of', 'repo_id_1'],
 ['user_id_4', 'is_member_of', 'repo_id_1'],
 ['user_id_24', 'is_member_of', 'repo_id_3'],
 ['user_id_5465', 'is_member_of', 'repo_id_3']]

Processing of followers table

In [9]:
!head {FOLLOWERS_PATH}

1,2,"2008-12-15 12:28:33"
1,4,"2008-12-15 12:28:33"
1,17896,"2008-12-15 12:28:33"
1,21523,"2008-12-15 12:28:33"
1,29121,"2008-12-15 12:28:33"
1,46833,"2008-12-15 12:28:33"
1,57004,"2008-12-15 12:28:33"
1,59446,"2012-08-21 10:26:17"
1,89393,"2010-08-23 23:53:39"
1,99896,"2013-05-03 04:18:40"


In [10]:
followers_rdd = sc.textFile(FOLLOWERS_PATH).map(lambda x: x.split(",")[:2]) \
    .map(lambda x: addIdentifiers(x, "user_id_", "user_id_")) \
    .map(swapColumns) \
    .map(lambda x: addRelation(x, FOLLOWS))

followers_rdd.take(5)

[['user_id_2', 'follows', 'user_id_1'],
 ['user_id_4', 'follows', 'user_id_1'],
 ['user_id_17896', 'follows', 'user_id_1'],
 ['user_id_21523', 'follows', 'user_id_1'],
 ['user_id_29121', 'follows', 'user_id_1']]

Processing of watchers table

In [11]:
!head {WATCHERS_PATH}

1,1,"2009-12-08 10:17:27"
1,2,"2009-12-08 10:17:27"
1,4,"2009-12-08 10:17:27"
1,6,"2010-02-05 06:35:04"
1,7,"2009-12-08 10:17:27"
1,8,"2010-03-26 04:57:36"
1,9,"2009-12-08 10:17:27"
1,10,"2009-12-08 10:17:27"
1,11,"2009-12-08 10:17:27"
1,12,"2009-12-08 10:17:27"


In [12]:
watchers_rdd = sc.textFile(WATCHERS_PATH).map(lambda x: x.split(",")[:2]) \
    .map(lambda x: addIdentifiers(x, "repo_id_", "user_id_")) \
    .map(swapColumns) \
    .map(lambda x: addRelation(x, WATCHES))

watchers_rdd.take(5)

[['user_id_1', 'watches', 'repo_id_1'],
 ['user_id_2', 'watches', 'repo_id_1'],
 ['user_id_4', 'watches', 'repo_id_1'],
 ['user_id_6', 'watches', 'repo_id_1'],
 ['user_id_7', 'watches', 'repo_id_1']]

### Split into train, validation and test datasets

Merge into one entity

In [13]:
rdd = members_rdd.union(followers_rdd).union(watchers_rdd)

sqlContext = SQLContext(sc)
df = sqlContext.createDataFrame(rdd, ['from_entity_id', 'relation', 'to_entity_id'])
df.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges')
shuffledDF = df.randomSplit([0.1, 0.1, 0.8])
val_set = shuffledDF[0]
test_set = shuffledDF[1]
train_set = shuffledDF[2]

Save into files

In [16]:
val_set.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges/validation')
test_set.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges/test')
train_set.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges/train')

### Store metadata

Metainfo we store in MongoDB

In [ ]:
def convert_dt(sdt):
    try:
        return pd.datetime.strptime(sdt, '%Y-%m-%d %H:%M:%S')
    except Exception as e:
        print(e)

#### Users

In [5]:
users_df = pd.read_csv(f'{DATA_FOLDER}users.csv',
                       header=None,
                       names=['id', 'login', 'company', 'created_at', 'type', 'fake', 'deleted', 'long', 'lat',
                              'country_code', 'state', 'city', 'location'],
#                        parse_dates=[3],
#                        quoting=1,
#                        quotechar='"',
#                        encoding='utf-8',
                       low_memory=False,
                       error_bad_lines=False).replace({'\\N': None}).set_index('id')
print(users_df.shape)
users_df['created_at'] = users_df['created_at'].map(convert_dt)
users_df = users_df[~users_df['created_at'].isna()]
users_df['fake'] = users_df['fake'].replace({1: True, 0: False, '1': True, '0': False}).astype(bool)
users_df['deleted'] = users_df['deleted'].replace({1: True, 0: False, '1': True, '0': False}).astype(bool)
users_df['long'] = users_df['long'].astype(float)
users_df['lat'] = users_df['lat'].astype(float)
users_df.shape

(30600249, 12)

In [3]:
mc['github']['users'].create_index('login')

'login_1'

In [6]:
to_insert = []

for uid, row in tn(users_df.iterrows(), total=users_df.shape[0]):
    to_insert.append({'_id': uid, **row.to_dict()})
    if len(to_insert) >= 10_000:
        mc['github']['users'].insert_many(to_insert)
        to_insert = []
if len(to_insert):
    mc['github']['users'].insert_many(to_insert)
mc['github']['users'].estimated_document_count()

30600249

Users example
![Users](./resources/images/Users.png)

#### Projects (repos)

In [ ]:
for field in ['name', 'owner_id', 'url']:
    mc['github']['projects'].create_index(field)

In [7]:
try:
    stop = False
    skip_count = 0
    for i in range(1000):
        print(i)
        repos_df = pd.read_csv(f'{DATA_FOLDER}projects.csv',
                               skiprows=10_000_000*i,
                               nrows=10_000_000,
                               header=None,
                               na_values=['\\N'],
                               names=["id", "url", "owner_id", "name", "description", "language",
                                      "created_at", "forked_from", "deleted", "updated_at", "?"],
                               error_bad_lines=False,
                               low_memory=False,
                              ).set_index('id')
        c = repos_df.shape[0]
        if c < 10_000_000:
            stop = True
        repos_df['created_at'] = repos_df['created_at'].map(convert_dt)
        repos_df = repos_df[~repos_df['created_at'].isna()]
        repos_df['updated_at'] = repos_df['updated_at'].map(convert_dt)
        repos_df = repos_df[~repos_df['updated_at'].isna()]
        skip_count += (c - repos_df.shape[0])
        repos_df.index = repos_df.index.astype(int)
        repos_df['url'] = repos_df['url'].astype(str).map(lambda x: x[29:])
        repos_df['deleted'] = repos_df['deleted'].replace({1: True, 0: False, 1: True, 0: False}).astype(bool)
        repos_df['owner_id'] = repos_df['owner_id'].astype(int)
        repos_df['forked_from'] = repos_df['forked_from'].astype(float)
        repos_df['?'] = repos_df['?'].astype(float)

        to_insert = []

        for rid, row in tn(repos_df.iterrows(), total=repos_df.shape[0]):
            to_insert.append({'_id': rid, **row.to_dict()})
            if len(to_insert) >= 10_000:
                mc['github']['projects'].insert_many(to_insert)
                to_insert = []
        if len(to_insert):
            mc['github']['project'].insert_many(to_insert)
        if stop:
            break
    m = f"[projects]: stored {mc['github']['projects'].estimated_document_count()} projects, {skip_count} skiped"
    tgn(m)
except Exception as e:
    m = f'[projects]: error - {e}'
    tgn(m)
m

'[projects]: stored 116010000 projects, 135185 skiped'

Repos example
![Projects](./resources/images/Projects.png)

## Training

![Partitions](./resources/images/Partitions.png)

*Figure 1.* The PBG partitioning scheme for large graphs. **Left:** nodes are divided into $P$ partitions that are sized to fit in memory. Edges are divided into buckets based on the partition of their source and destination nodes. In distributed mode, multiple buckets with non-overlapping partitions can be executed in parallel (red squares). **Center:** Entity types with small cardinality do not have to be partitioned; if all entity types used for tail nodes are unpartitioned, then edges can be divided into $P$ buckets based only on source node partitions. **Right:** the ‘inside-out’ bucket order guarantees that buckets have at least one previously-trained embedding partition. Empirically, this ordering produces better embeddings than other alternatives (or random)

![Architecture](./resources/images/Architecture.png)

*Figure 2.* Memory-efficient batched negative sampling. Embeddings are fetched for the $B$ source and destination entities in a batch of edges, as well as $B$ uniformly-sampled source and destination entities. Each chunk of $B_n/2$ edges is corrupted with all source or destination entities in its chunk, as well as the corresponding chunk of the uniform embeddings, resulting in $B_n$ negative examples per positive edge. The negative scores are computed via a batch matrix multiply.

Lerer, Adam, et al. "PyTorch-BigGraph: A Large-scale Graph Embedding System." arXiv preprint arXiv:1903.12287 (2019).
https://arxiv.org/pdf/1903.12287.pdf

Clear dir from possible previous training (or archive configs)

In [14]:
for TEMPLATE in ['entity_count_all_*.txt', 'graph-*_partitioned/*', 'dictionary.json', 'dynamic_rel_count.txt']:
    !rm -f {DATA_FOLDER}graph_edges/{TEMPLATE}

See `graph_config.py` config file for details (`torchbiggraph_train -h` for help)

Let's exclude 'long tail' where entity (repo/user) has less than {ENTITY_MIN_COUNT} relations

In [5]:
ENTITY_MIN_COUNT = 30

Creating needed partitions

In [6]:
!cat graph_config.py

#!/usr/bin/env python3
# BASE_PATH = "./data/dump/graph_edges"
BASE_PATH = "./data/mysql-2021-03-06/graph_edges"

def get_torchbiggraph_config():

    config = dict(
        # I/O data
        entity_path=BASE_PATH,
        edge_paths=[f'{BASE_PATH}/edges_partitioned'],  # [f'{BASE_PATH}/graph-train_partitioned'],
        checkpoint_path=BASE_PATH,

        # Graph structure
        entities={
            'all': {'num_partitions': 4},
        },
        relations=[{
            'name': 'all_edges',
            'lhs': 'all',
            'rhs': 'all',
            'operator': 'complex_diagonal',
        }],
        dynamic_relations=True,

        # Scoring model
        dimension=100,
        global_emb=False,
        comparator='dot',

        # Training
        num_epochs=10,
        num_uniform_negs=100,
        loss_fn='softmax',
        lr=0.1,
        regularization_coef=1e-3,

        # Evaluation during training
        eval_fraction=0,  # to reproduce results, we need to use all

In [10]:
!head data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000.csv

from_entity_id	relation	to_entity_id
user_id_1	is_member_of	repo_id_1
user_id_2	is_member_of	repo_id_1
user_id_4	is_member_of	repo_id_1
user_id_24	is_member_of	repo_id_3
user_id_5465	is_member_of	repo_id_3
user_id_11834	is_member_of	repo_id_3
user_id_11983	is_member_of	repo_id_3
user_id_17241	is_member_of	repo_id_3
user_id_26333	is_member_of	repo_id_3


In [7]:
with open("./data/mysql-2021-03-06/graph_edges/dictionary.json") as f:
    d = json.load(f)
d["relations"]

['follows', 'watches', 'is_member_of']

In [8]:
pd.Series(d["entities"]["all"]).str[:4].value_counts()

user    1385786
repo     594979
dtype: int64

In [12]:
%time !torchbiggraph_import_from_tsv --lhs-col=0 --rel-col=1 --rhs-col=2 --entity-min-count={ENTITY_MIN_COUNT} graph_config.py {DATA_FOLDER}graph_edges/*.csv > logs/formating.log

CPU times: user 13.9 s, sys: 2.59 s, total: 16.5 s
Wall time: 22min 14s


In [13]:
!head -n 20 logs/formating.log

Looking up relation types in the edge files...
- Found 4 relation types
- Removing the ones with fewer than 1 occurrences...
- Left with 4 relation types
- Shuffling them...
Searching for the entities in the edge files...
Entity type all:
- Found 37508168 entities
- Removing the ones with fewer than 30 occurrences...
- Left with 1980765 entities
- Shuffling them...
Preparing entity path ./data/mysql-2021-03-06/graph_edges:
- Writing count of entity type all and partition 0
- Writing count of entity type all and partition 1
- Writing count of entity type all and partition 2
- Writing count of entity type all and partition 3
- Writing count of dynamic relations
Preparing edge path ./data/mysql-2021-03-06/graph_edges/part-00000-0f2a1713-643a-4216-9efb-4e141b286eb2-c000_partitioned, out of the edges found in ./data/mysql-2021-03-06/graph_edges/part-00000-0f2a1713-643a-4216-9efb-4e141b286eb2-c000.csv
- Edges will be partitioned in 4 x 4 buckets.
- Processed 100000 edges so far...


In [14]:
for TEMPLATE in ['checkpoint_version.txt', 'embeddings_all_*.h5', 'model.v*.h5']:
    !rm -f {DATA_FOLDER}graph_edges/{TEMPLATE}

Actually, training

In [ ]:
# %time !torchbiggraph_train graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/graph-train_partitioned > logs/train.log

In [14]:
%time !torchbiggraph_train graph_config.py > logs/train.log

CPU times: user 5min 17s, sys: 49.7 s, total: 6min 6s
Wall time: 4h 46min


In [15]:
!head logs/train.log
print('-'*100)
!tail logs/train.log

2022-07-02 01:36:40,601   [Trainer-0] Loading entity counts...
2022-07-02 01:36:40,602   [Trainer-0] Creating workers...
2022-07-02 01:36:40,659   [Trainer-0] Initializing global model...
2022-07-02 01:36:40,662   [Trainer-0] Starting epoch 1 / 10, edge path 1 / 1, edge chunk 1 / 1
2022-07-02 01:36:40,662   [Trainer-0] Edge path: ./data/mysql-2021-03-06/graph_edges/edges_partitioned
2022-07-02 01:36:40,662   [Trainer-0] still in queue: 15
2022-07-02 01:36:40,662   [Trainer-0] Swapping partitioned embeddings None ( 3 , 3 )
2022-07-02 01:36:40,662   [Trainer-0] ( 3 , 3 ): Loading entities
2022-07-02 01:38:31,335   [Trainer-0] ( 3 , 3 ): bucket 1 / 16 : Processed 10360238 edges in 109.53 s ( 0.095 M/sec ); io: 1.14 s ( 391.50 MB/sec )
2022-07-02 01:38:31,335   [Trainer-0] ( 3 , 3 ): loss:  6.99258 , violators_lhs:  21.6603 , violators_rhs:  15.4519 , count:  10360238
----------------------------------------------------------------------------------------------------
2022-07-02 06:22:36,38

## Evaluating

### BigGraph Evaluation

Offline evaluation in the BigGraph performed using `torchbiggraph_eval` that samples some number of negative edges with a particular positive edge. In practice, since we are using sparse graphs - this implementation assumes that randomly given edge is a negative one with high probability. Therefore some positive edges may be mixed with a negative one.

Several metrics are built-in into the library and calculated for each batch separately, and afterwards globally, such as:
 - *Mean Rank (pos_rank)* - the average of the ranks of all positives (lower is better, best is 1).
 - *Mean Reciprocal Rank (MRR)* - the average of the reciprocal of the ranks of all positives (higher is better, best is 1).
 $$\dfrac{1}{n} \sum_{i=1}^{n} \dfrac{1}{rank}, $$
 where $n$ is the number of all positives, and $rank$ is rank of a positive result in a sample
 - *r1* - the fraction of positives that rank better than all their negatives, i.e., have a rank of 1 (higher is better, best is 1).
 - *r10* - the fraction of positives that rank in the top 10 among their negatives (higher is better, best is 1).
 - *r50* - the fraction of positives that rank in the top 50 among their negatives (higher is better, best is 1).
 - *AUC* - an estimation of the probability that a randomly chosen positive scores higher than a randomly chosen negative (any negative, not only the negatives constructed by corrupting that positive).

**Evaluating on train**

In [16]:
!torchbiggraph_eval graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/edges_partitioned > logs/eval_train.log

In [17]:
!head logs/eval_train.log
print('-'*100)
!tail logs/eval_train.log

2022-07-02 06:22:44,155   [Evaluator] Starting edge path 1 / 1 (./data/mysql-2021-03-06/graph_edges/edges_partitioned)
2022-07-02 06:23:09,296   [Evaluator] ( 0 , 0 ): Processed 10618701 edges in 25 s (0.43M/sec); load time: 0.6 s
2022-07-02 06:23:09,296   [Evaluator] Stats for edge path 1 / 1, bucket ( 0 , 0 ): pos_rank:  23.7711 , mrr:  0.243724 , r1:  0.164991 , r10:  0.382086 , r50:  0.907372 , auc:  0.863558 , count:  10618701
2022-07-02 06:23:32,968   [Evaluator] ( 0 , 1 ): Processed 10391397 edges in 23 s (0.44M/sec); load time: 0.25 s
2022-07-02 06:23:32,968   [Evaluator] Stats for edge path 1 / 1, bucket ( 0 , 1 ): pos_rank:  23.1709 , mrr:  0.253796 , r1:  0.175583 , r10:  0.392196 , r50:  0.918345 , auc:  0.867914 , count:  10391397
2022-07-02 06:23:56,670   [Evaluator] ( 0 , 2 ): Processed 10475272 edges in 23 s (0.45M/sec); load time: 0.25 s
2022-07-02 06:23:56,670   [Evaluator] Stats for edge path 1 / 1, bucket ( 0 , 2 ): pos_rank:  23.3282 , mrr:  0.250947 , r1:  0.17266

**Evaluating on validation**

In [5]:
!torchbiggraph_eval graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/graph-val_partitioned > logs/eval_val.log

In [6]:
!head logs/eval_val.log
print('-'*100)
!tail logs/eval_val.log

2019-08-14 18:12:50  Starting edge path 1 / 1 (./data/mysql-2019-06-01/graph_edges/graph-val_partitioned)
2019-08-14 18:12:51  ( 0 , 0 ): Processed 799176 edges in 1.7 s (0.48M/sec); load time: 0.12 s
2019-08-14 18:12:51  Stats for edge path 1 / 1, bucket ( 0 , 0 ): pos_rank:  9.97506 , mrr:  0.388542 , r1:  0.236046 , r10:  0.720194 , r50:  0.97284 , auc:  0.943282 , count:  799176
2019-08-14 18:12:53  ( 0 , 1 ): Processed 822664 edges in 1.6 s (0.52M/sec); load time: 0.1 s
2019-08-14 18:12:53  Stats for edge path 1 / 1, bucket ( 0 , 1 ): pos_rank:  9.79906 , mrr:  0.39167 , r1:  0.23917 , r10:  0.723014 , r50:  0.974304 , auc:  0.944365 , count:  822664
2019-08-14 18:12:55  ( 0 , 2 ): Processed 810187 edges in 1.6 s (0.52M/sec); load time: 0.084 s
2019-08-14 18:12:55  Stats for edge path 1 / 1, bucket ( 0 , 2 ): pos_rank:  9.51358 , mrr:  0.397661 , r1:  0.243526 , r10:  0.731363 , r50:  0.975832 , auc:  0.946319 , count:  810187
2019-08-14 18:12:56  ( 0 , 3 ): Processed 819618 edges

**Result table**

| Metric   | Train | Validation | Train_new |
|----------|-------|------------|-----------|
| pos_rank | 6.77  | 10.05      | 24.148    |
| MRR      | 0.48  | 0.38       | 0.240351  |
| r1       | 0.32  | 0.23       | 0.160709  |
| r10      | 0.81  | 0.71       | 0.379815  |
| r50      | 0.99  | 0.97       | 0.898278  |
| AUC      | 0.97  | 0.94       | 0.859187  |

### User evaluation

Besides that, we performed user evaluation. We engaged users that have not heard anything about our project and actively using open source projects and GitHub. As a setting for the experiment, we asked to provide five repositories that users familiar with and proposed to them two recommendations for each. First one is our best guess from Annoy nearest neighbour search and second one - random repository with 1000+ stars. Please see the table with the experiment data:

| Provided repo | Our suggestion    | Random repo                         | Choice                            |
|---------------|-------------------|-------------------------------------|-----------------------------------|
| yarnpkg/yarn  | prettier/prettier | fossasia/sphinx_fossasia_theme      | both irrelevant                   |
| npm/npm       | gulpjs/gulp       | lhartikk/ArnoldC                    | our suggestion marked as relevant |
| rspec/rspec   | rspec/rspec-core  | FLIF-hub/FLIF                       | both irrelevant                   |
| rails/rails   | sinatra/sinatra   | javamelody/javamelody               | our suggestion marked as relevant |
| ruby/ruby     | rails/rails       | ustjavac/Programming-Alpha-To-Omega | both irrelevant                   |

As we see in all cases, users correctly identified random repositories and marked 40% of our suggestions as useful, which is similar to the rank1 metric on the validation set. However, we cannot say that this result is statistically significant due to small sample size. 

Besides that, one user asked to search for repos that are similar to one particular repository for his current project task and replied that the top 10 suggestions contained "interesting results that I have not found googling".

## Prepare tensorboard

In [ ]:
# !torchbiggraph_export_to_tsv --dict {DATA_FOLDER}/graph_edges/dictionary.json \
# --checkpoint {DATA_FOLDER}/graph_edges/ --out {DATA_FOLDER}/graph_edges/embeddings.tsv

Slow ↑ method

In [18]:
import json
import h5py
import os

!mkdir -p tb
!mkdir -p tb/embeddings

def dropCategories(df, col, threshold):
    drop_list = df[col].value_counts()[threshold:].index.to_list()
    df[col].cat.remove_categories(drop_list, inplace=True)
    return df

For faster results let's see only on repos with 1K+ stars and users with 100+ followers/following

In [3]:
good_rids = pd.read_pickle('./tb/embeddings_old/repos_19k_gte1k.pkl').index.tolist()
len(good_rids)

18670

In [19]:
good_rids = pd.read_csv("repos_gte1k.txt", names=[0])[0].tolist()
len(good_rids)

27367

In [4]:
good_uids = pd.read_pickle('./tb/embeddings_old/users_following_29k_gte100.pkl').index.tolist()
good_uids += pd.read_pickle('./tb/embeddings_old/users_followers_24k_gte100.pkl').index.tolist()
good_uids = list(set(good_uids))
len(good_uids)

47666

In [20]:
good_uids = pd.read_csv("users_gte500.txt", names=[0])[0].tolist()
len(good_uids)

74372

Read metadata

In [21]:
repos_cur = mc['github']['projects'].find({"_id": {"$in": good_rids}}, ['url', 'language', 'created_at', 'updated_at'])
repos_df = pd.DataFrame(list(repos_cur)).set_index('_id')[['url', 'language', 'created_at', 'updated_at']]
repos_df.language = repos_df.language.astype("category")
repos_df = dropCategories(repos_df, "language", 42)
repos_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24963 entries, 6 to 127683206
Data columns (total 4 columns):
url           24963 non-null object
language      18637 non-null category
created_at    24963 non-null datetime64[ns]
updated_at    24963 non-null datetime64[ns]
dtypes: category(1), datetime64[ns](2), object(1)
memory usage: 806.0+ KB


In [22]:
repos_df.sample(5)

,url,language,created_at,updated_at
_id,,,,
32683738,griddb/griddb_nosql,C++,2016-02-24 06:31:40,2019-02-26 09:03:12
121196295,amusi/awesome-lane-detection,NaN,2018-12-26 08:08:05,2019-02-27 02:12:52
59181710,HowProgrammingWorks/Index,NaN,2017-03-09 00:12:29,2019-02-23 19:12:42
3708578,iamcal/emoji-data,HTML,2013-05-14 03:01:12,2019-02-26 07:37:36
49949379,frappe/gantt,JavaScript,2016-08-26 12:17:58,2019-02-26 09:45:44


In [23]:
repos_df["language"].value_counts(dropna=False)

NaN                 6326
JavaScript          4719
Python              2216
Java                2084
Go                  1142
C++                 1011
C                    880
PHP                  852
Objective-C          784
Ruby                 672
HTML                 587
C#                   537
Swift                516
Shell                455
CSS                  435
TypeScript           266
Rust                 194
Jupyter Notebook     144
VimL                 142
CoffeeScript         116
Scala                113
Clojure               70
Kotlin                64
Vue                   61
Lua                   57
Haskell               52
Emacs Lisp            50
Perl                  44
TeX                   44
Makefile              38
R                     35
PowerShell            35
Dart                  33
Elixir                31
Erlang                28
Assembly              25
Groovy                25
OCaml                 24
Objective-C++         13
Matlab                13


In [24]:
ufields = ['login', 'type', 'fake', 'location', 'deleted', 'country_code', 'created_at']
users_cur = mc['github']['users'].find({"_id": {"$in": good_uids}}, ufields)
users_df = pd.DataFrame(list(users_cur)).set_index('_id')[ufields]
users_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 73131 entries, 2 to 49715417
Data columns (total 7 columns):
login           73131 non-null object
type            73131 non-null object
fake            73131 non-null bool
location        50007 non-null object
deleted         73131 non-null bool
country_code    43771 non-null object
created_at      73131 non-null datetime64[ns]
dtypes: bool(2), datetime64[ns](1), object(4)
memory usage: 3.5+ MB


In [25]:
users_df.sample(5)

,login,type,fake,location,deleted,country_code,created_at
_id,,,,,,,
7258074,bskinn,USR,False,"Dayton, OH, USA",False,None,2015-03-05 02:52:51
1756027,konapun,USR,False,"Minneapolis, MN",False,us,2010-11-27 02:25:06
3747423,nailuoGG,USR,False,杭州,False,cn,2013-05-28 23:13:03
576606,brunogasparetto,USR,False,None,False,None,2012-04-05 21:13:11
96134,MariusRugan,USR,False,Amsterdam,False,nl,2009-10-24 05:57:27


In [26]:
path = f'{DATA_FOLDER}graph_edges/'
files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.find("embeddings_all") != -1])
files = files[0:2] + files[8:] + files[2:8]
files

['./data/mysql-2021-03-06/graph_edges/embeddings_all_0.v10.h5',
 './data/mysql-2021-03-06/graph_edges/embeddings_all_1.v10.h5',
 './data/mysql-2021-03-06/graph_edges/embeddings_all_2.v10.h5',
 './data/mysql-2021-03-06/graph_edges/embeddings_all_3.v10.h5']

Next actions need some RAM!

In [27]:
with open(os.path.join(path, "dictionary.json"), "rt") as tf:
    dictionary = json.load(tf)
print(dictionary['relations'])
all_entities = dictionary['entities']['all']
len(all_entities)

['follows', 'watches', 'is_member_of']


1980765

In [28]:
pd.Series(all_entities).map(lambda x: x.split('_id_')[0]).value_counts()

user    1385786
repo     594979
dtype: int64

Read embeddings

In [29]:
embeddings=[]
for file in tn(files):
    with h5py.File(file, "r") as hf:
        embedding = hf["embeddings"][...]
    embeddings.append(embedding)
embeddings = np.concatenate(embeddings, axis=0)
embeddings.shape

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:2: TqdmDeprecationWarning:

This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`



  0%|          | 0/4 [00:00<?, ?it/s]

(1980765, 100)

In [30]:
emb_df = pd.Series({i: e for i, e in tn(zip(all_entities, embeddings))}).to_frame(name='embeddings')
emb_df.shape

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:1: TqdmDeprecationWarning:

This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`



0it [00:00, ?it/s]

(1980765, 1)

Combine all metainfo with embeddings

In [31]:
repos_emb_df = pd.Series({int(i.split('_')[-1]): e for i, e in emb_df.loc[[f'repo_id_{i}' for i in repos_df.index]]['embeddings'].iteritems()}).to_frame(name='embeddings')
repos_df = repos_df.join(repos_emb_df)
repos_df.sample(5)

,url,language,created_at,updated_at,embeddings
_id,,,,,
6026040,krschultz/android-proguard-snippets,NaN,2013-10-22 16:44:36,2019-02-26 21:45:02,"[0.12167536, -0.39032668, 0.06987032, -0.16309..."
60279947,cujanovic/SSRF-Testing,NaN,2017-03-21 07:57:51,2019-02-26 20:07:48,"[0.13018847, -0.7370862, 0.0031132405, -0.0389..."
47834930,Developer-Y/Scalable-Software-Architecture,NaN,2016-10-08 05:39:39,2019-02-27 04:10:52,"[-0.13821863, 0.09080807, 0.09075221, -0.05715..."
13460722,thoas/picfit,Go,2014-12-06 17:30:45,2019-02-26 13:19:05,"[-0.03072633, 0.016448885, 0.007876763, -0.077..."
11763541,davidtheclark/scalable-css-reading-list,NaN,2014-09-03 00:13:24,2019-02-21 08:49:19,"[-0.01716408, 0.033613782, 0.046365477, -0.098..."


In [32]:
users_emb_df = pd.Series({int(i.split('_')[-1]): e for i, e in emb_df.loc[[f'user_id_{i}' for i in users_df.index]]['embeddings'].iteritems()}).to_frame(name='embeddings')
users_df = users_df.join(users_emb_df)
users_df.sample(5)

,login,type,fake,location,deleted,country_code,created_at,embeddings
_id,,,,,,,,
32292273,sessionstop,USR,False,Undefined,False,None,2013-04-15 11:57:52,"[-0.051701285, -0.3590143, -0.1335563, 0.17071..."
3325023,4406arthur,USR,False,Taiwan,False,tw,2013-07-23 07:43:08,"[-0.010585736, 0.03796979, -0.04421862, 0.0871..."
9540398,nelsera,USR,False,"Sao Paulo, Brazil",True,br,2013-12-15 01:29:09,"[0.119962566, 0.47066247, 0.11077086, -0.16136..."
12563374,soyemi,USR,False,Shanghai,False,cn,2015-10-27 15:33:53,"[-0.16574927, -0.525374, -0.007845831, 0.06387..."
42282110,dbrunonascimento,USR,False,None,False,None,2018-03-14 17:20:15,"[-0.107096314, 0.30566052, 0.27802256, 0.08349..."


Save for TensorBoard

In [36]:
repos_df.to_pickle("./tb/embeddings/repos_25k_gte1k.pkl")

In [48]:
users_df["embeddings"].map(len).value_counts(dropna=False)

100    73131
Name: embeddings, dtype: int64

In [50]:
users_df["type"].value_counts(dropna=False)

USR    72996
ORG      135
Name: type, dtype: int64

In [51]:
users_df.to_pickle("./tb/embeddings/users_73k_gte500.pkl")

If trained with `global = True` - embeddings for different types could be comparable (same hyperspace)

In [ ]:
# repos_df.index = repos_df.index.map(lambda x: f'r{x}')
# users_df.index = users_df.index.map(lambda x: f'u{x}')
# combined_df = pd.concat([repos_df, users_df])
# combined_df.to_pickle("./tb/embeddings/repos_19k_gte1k_users_48k_gte100.pkl")

↑ embeddings for repos and users are far away from each other - no sense in viewing into combined multidimentional space

(reason - no `global = True` in config for better independent representations)

In [40]:
!rm -rf ./tb/embeddings/.ipynb_checkpoints

Now, `cd tb` & `run2.sh` at terminal with pre-installed docker or read `tb/README.md`

Then, open `HOSTNAME:PORT`, where `HOSTNAME` cane from machine, where the last script ran and `PORT` = `8002` (by default, but configurable)

## Building Annoy (library to do nearest neighbor search)

In [52]:
from annoy import AnnoyIndex

### Repos

Read dataframe

In [8]:
repos_df = pd.read_pickle("./tb/embeddings/repos_19k_gte1k.pkl")
repos_df.shape

(18670, 5)

Dimension

In [9]:
dim = len(repos_df['embeddings'].iloc[0])
dim

100

Mapping for index

In [10]:
repos_ann = AnnoyIndex(dim, 'angular')  # Length of item vector that will be indexed
repos_mapping = {}
for i, (repo_id, e) in enumerate(repos_df['embeddings'].iteritems()):
    repos_ann.add_item(i, list(e))
    repos_mapping[repo_id] = i

Build index

In [11]:
n_trees = 1_000

%time repos_ann.build(n_trees)

CPU times: user 38.7 s, sys: 25.3 ms, total: 38.8 s
Wall time: 38.8 s


True

Save index

In [12]:
repos_ann.save('./data/repos.ann')
pd.to_pickle(repos_mapping, './data/repos_mapping.pkl')

### Users

Read dataframe

In [13]:
users_df = pd.read_pickle("./tb/embeddings/users_48k_gte100.pkl")
users_df.shape

(47666, 8)

Dimension

In [14]:
dim = len(users_df['embeddings'].iloc[0])
dim

100

Mapping for index

In [15]:
users_ann = AnnoyIndex(dim, 'angular')
users_mapping = {}
for i, (user_id, e) in enumerate(users_df['embeddings'].iteritems()):
    users_ann.add_item(i, list(e))
    users_mapping[user_id] = i

Build index

In [16]:
n_trees = 1_000

%time users_ann.build(n_trees)

CPU times: user 1min 54s, sys: 147 ms, total: 1min 54s
Wall time: 1min 54s


True

Save index

In [17]:
users_ann.save('./data/users.ann')
pd.to_pickle(users_mapping, './data/users_mapping.pkl')

## Recommender search example

### Repos

Read dataframe

In [54]:
repos_df = pd.read_pickle("./tb/embeddings/repos_19k_gte1k_old.pkl")
repos_df.shape

(18670, 5)

Dimension

In [55]:
dim = len(repos_df['embeddings'].iloc[0])
dim

100

Read index & mapping

In [56]:
repos_ann = AnnoyIndex(dim, 'angular')
repos_ann.load('./data/repos.ann') # super fast, will just mmap the file
repos_mapping = pd.read_pickle('./data/repos_mapping.pkl')
repos_mapping_rev = {v: k for k, v in repos_mapping.items()}
len(repos_mapping)

18670

In [57]:
def get_nns_by_repo_id(repo_id, n_nn=10):
    res, dist = repos_ann.get_nns_by_item(repos_mapping[repo_id], n_nn, include_distances=True) # will find the n nearest neighbors
    res = [repos_mapping_rev[r] for r in res]
    res = repos_df.loc[res]
    res['distance'] = [round(d, 3) for d in dist]
    return res

Type interesting query for repos search (don't forget about unpopularity limitation)

In [58]:
repo_query = 'apache/spark'

repos_df[repos_df['url'].str.lower().str.contains(repo_query)]

,url,language,created_at,updated_at,embeddings
_id,,,,,
8196280,apache/spark,Scala,2014-02-25 08:00:08,2019-02-26 23:57:18,"[0.5067623, 0.013571289, -0.07234435, -0.00304..."


In [60]:
repo_query = 'jaeger'

repos_df[repos_df['url'].str.lower().str.contains(repo_query)]

,url,language,created_at,updated_at,embeddings
_id,,,,,
49690308,uber/jaeger,NaN,2016-04-15 18:49:02,2019-02-17 22:47:07,"[1.3138223, -0.042256385, 0.08451154, 0.102117..."
75368828,jaegertracing/jaeger,Go,2016-04-15 18:49:02,2019-02-26 12:31:17,"[0.9364147, -0.03269517, -0.05895982, -0.03663..."


Choose needed one to search nearest neighbors

In [74]:
repo_id = 75368828

get_nns_by_repo_id(repo_id)

,url,language,created_at,updated_at,embeddings,distance
_id,,,,,,
75368828,jaegertracing/jaeger,Go,2016-04-15 18:49:02,2019-02-26 12:31:17,"[0.9364147, -0.03269517, -0.05895982, -0.03663...",0.000
51981313,istio/istio,NaN,2016-11-18 23:57:21,2019-02-26 23:18:37,"[0.81216884, 0.022117646, -0.02624824, 0.01300...",0.472
55517952,linkerd/linkerd,Scala,2016-01-13 22:14:52,2019-02-27 01:08:33,"[1.1030256, 0.04486563, 0.0943077, 0.11222585,...",0.507
74717642,envoyproxy/envoy,C++,2016-08-08 15:07:24,2019-02-27 04:26:08,"[0.86620814, 0.03407981, -0.057631254, 0.01349...",0.522
20271797,openzipkin/zipkin,Scala,2012-06-06 18:26:16,2019-02-26 10:59:17,"[0.9452616, -0.06330491, -0.06207261, -0.09240...",0.530
90114548,vitessio/vitess,Go,2013-06-27 21:20:28,2019-02-27 09:40:09,"[0.83330035, 0.04869755, -0.055808663, -0.0239...",0.549
29185598,TykTechnologies/tyk,Go,2014-05-07 14:38:24,2019-02-26 19:46:33,"[1.0778252, 0.017363897, -0.023363054, -0.0040...",0.554
2363758,prometheus/prometheus,Go,2012-11-24 11:14:12,2019-02-26 10:10:23,"[0.838226, 0.025573649, -0.012193335, 0.034605...",0.557
103337857,linkerd/linkerd2,Rust,2018-07-09 00:47:00,2019-02-27 03:12:19,"[0.8537553, 0.06280027, -0.20727564, -0.023235...",0.561


All Big Data, distributed tools

### Users

Read dataframe

In [24]:
users_df = pd.read_pickle("./tb/embeddings/users_48k_gte100.pkl")
users_df.shape

(47666, 8)

Dimension

In [25]:
dim = len(users_df['embeddings'].iloc[0])
dim

100

Read index & mapping

In [64]:
users_ann = AnnoyIndex(dim, 'angular')
users_ann.load('./data/users.ann') # super fast, will just mmap the file
users_mapping = pd.read_pickle('./data/users_mapping.pkl')
users_mapping_rev = {v: k for k, v in users_mapping.items()}
len(users_mapping)

47666

In [65]:
def get_nns_by_user_id(user_id, n_nn=10):
    res, dist = users_ann.get_nns_by_item(users_mapping[user_id], n_nn, include_distances=True) # will find the n nearest neighbors
    res = [users_mapping_rev[r] for r in res]
    res = users_df.loc[res]
    res['distance'] = [round(d, 3) for d in dist]
    return res

Type interesting query for users search (don't forget about unpopularity limitation)

In [66]:
user_query = 'wiki'

users_df[users_df['login'].str.lower().str.contains(user_query)]

,login,type,fake,location,deleted,country_code,created_at,embeddings
_id,,,,,,,,
3475926,wikibootup,USR,False,None,False,None,2014-01-23 06:04:25,"[-0.065740615, -0.23684335, 0.16004804, -0.155..."
7033202,wikijm,USR,False,France,False,fr,2015-01-27 11:18:54,"[-0.0124398125, -0.33704513, -0.038215928, -0...."


Choose needed one to search nearest neighbors

In [71]:
user_id = 1652246

get_nns_by_user_id(user_id)

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:4: FutureWarning:


Passing list-likes to .loc or [] with any missing label will raise
KeyError in the future, you can use .reindex() as an alternative.

See the documentation here:
https://pandas.pydata.org/pandas-docs/stable/indexing.html#deprecate-loc-reindex-listlike



,login,type,fake,location,deleted,country_code,created_at,embeddings,distance
_id,,,,,,,,,
1652246,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.000
3617048,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.839
286093,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.853
182762,dalinaum,USR,False,The Peach Blossom Spring,False,None,2009-10-28 01:13:40,"[-0.05793532, 0.34396824, -0.09501341, 0.06405...",0.873
884777,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.894
5712011,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.909
13295853,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.912
68757,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.915
68445,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.937


Korean "cluster"

If we go for their profiles - some similar stuff could be found there

It is essential to remember about absolute distance, not just TOP N, because some entities have neighbors close to them, while others could have not